In [ ]:
pip install -U pydantic

In [ ]:
import json
import torch
import numpy as np

In [ ]:
from deeppavlov import build_model, train_model, train_evaluate_model_from_config, evaluate_model
from deeppavlov.core.common.file import read_json

In [ ]:
from deeppavlov.dataset_readers.hallucination_detection_reader import HallucinationDatasetReader, RAGTruthDatasetReader

In [ ]:
from transformers import AutoModel, AutoConfig, AutoTokenizer, AutoModelForTokenClassification

In [ ]:
from deeppavlov.core.data.data_learning_iterator import DataLearningIterator
from deeppavlov.models.preprocessors.torch_transformers_preprocessor import TorchTransformersHallucinationDetectorPreprocessor 
from deeppavlov.metrics.fmeasure import token_binary_f1, token_binary_precision, token_binary_recall

In [ ]:
device = 'cuda'

In [ ]:
path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_large.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_mbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_deberta_small.json'


# path = 'deeppavlov/configs/hallucination_detection/ragtruth_eurobert_base.json'

In [ ]:
config = read_json(path)

In [ ]:
model = build_model(config, load_trained=True)

In [16]:
contexts = ["France is a country in Europe. The capital of France is Paris. The population of France is 67 million.",]
question = "What is the capital of France? What is the population of France?"
answer = "The capital of France is Paris. The population of France is 69 million."

In [2]:
sample = {
    'context': contexts,
    'question': question,
    'answer': answer,
    "labels": [],
}

In [ ]:
out = model([sample])

In [ ]:
spans = out[-1][0]

In [ ]:
spans

In [24]:
import requests

url = "http://localhost:5000/model"
response = requests.post(url, json={"x": [sample]})
print(response.json()[-1])

KeyError: -1

In [43]:
import requests

url = "http://localhost:5000/model"
# url = "https://7008.deeppavlov.ai/model"

input = {
    "context_raw": ["France is a country in Europe. The capital of France is Paris. The population of France is 67 million."],
    "question_raw": ["What is the capital of France? What is the population of France?"], 
    "answer_raw": ["The capital of France is Paris. The population of France is 69 million."]
}

response = requests.post(url, json=input)
print(response.json())
response

[[' The population of France is 69 million.', 31, 0.8841314315795898]]


<Response [200]>

In [9]:
probabilities = out[1]

NameError: name 'out' is not defined

In [ ]:
_, _, offsets, answer_start_token = TorchTransformersHallucinationDetectorPreprocessor.prepare_tokenized_input(
    self.tokenizer, sample['prompt'], sample['answer'], self.max_seq_length
)

In [ ]:
out[0]